# A Tool-Calling Travel Agent

A LangGraph agent with seven tools — flights, hotels, restaurants, weather,
currency, an FAQ retriever, and an itinerary planner — over live APIs (Amadeus,
Tavily) and a LanceDB knowledge base built from a parsed travel guide.

**Evaluated on 14 conversational scenarios, half of them in Persian.** The agent
resolves relative dates ("next week", "next month") against the current date,
maps free-text city names to IATA codes through fuzzy matching, calls tools in
parallel when a request needs several, and answers in the language it was asked
in.

Selected results:

- *"پرواز تهران دبی"* (Persian, no date) → `search_flights(Tehran, Dubai, 2026-02-14)`,
  IKA→DXB, 3 offers. It also tried IKA→DWC, got an empty list, and dropped it.
- *"tokyo to newyork next month flight"* → resolved to `2026-03-14`, expanded to
  2 origin and 3 destination airports, checked all 6 pairs, collected 18 offers.
- *"i want to go dubai from tehran next week find hotels and flight and check
  weather and also currency exchange"* → **four tools called in parallel** in one
  turn.

The honest finding is in `search_faq`: a Persian FAQ query retrieved a chunk about
language barriers instead of travel documents — a retrieval failure the LLM then
covered for by answering correctly from its own knowledge. Full scenario-by-scenario
analysis in [`docs/travelbot-evaluation.md`](../docs/travelbot-evaluation.md).

In [ ]:
from pathlib import Path

# Data is resolved relative to the repository root, so the notebook runs the
# same whether Jupyter was started here or one level up.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'

## 1. Credentials

All API keys come from the environment — see the README for the variable names and
where to get each one. Nothing is committed.

This cell also constructs the Amadeus client that the flight and hotel tools use.

In [ ]:
import os
from amadeus import Client

# Credentials are read from the process environment. Export these before
# starting Jupyter; see the README for what each one is and where to get it.
REQUIRED = [
    "LLM_API_KEY",            # OpenAI-compatible chat endpoint
    "AMADEUS_CLIENT_ID",      # flights and hotels
    "AMADEUS_CLIENT_SECRET",
    "TAVILY_API_KEY",         # web search
]
# LLAMA_CLOUD_API_KEY is only needed to re-parse the travel guide PDF. The
# parsed output is committed as data/parsed_travel_book.csv, so the FAQ tool
# works without it.

missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise RuntimeError(
        "Missing environment variables: " + ", ".join(missing)
        + "\nExport them in your shell before launching Jupyter."
    )

# The flight and hotel tools below use this client.
amadeus = Client(
    client_id=os.environ["AMADEUS_CLIENT_ID"],
    client_secret=os.environ["AMADEUS_CLIENT_SECRET"],
)
print("Credentials found; Amadeus client ready.")

## 2. Reference data

Two Kaggle datasets, downloaded at run time rather than committed: IATA airport
codes and a country-to-currency mapping.

In [ ]:
import kagglehub

iata_csv = kagglehub.dataset_download("zinovadr/iata-airport-code") + "/airport-codes_csv.csv"

currency_csv = kagglehub.dataset_download("phanee16/currency-and-country-code-mapping") + "/country_code_to_currency_code.csv"

### City to IATA codes

Building a `city -> [airport codes]` map. One city can have several airports
(`tokyo -> [NRT, HND]`, `new york -> [JFK, EWR, LGA]`), so the value is a list and
the flight tool searches every pair — which is why the Tokyo→New York query below
checks six routes.

Tehran is hardcoded to `IKA` because the dataset lists several Tehran airports and
only Imam Khomeini International handles international traffic.

In [ ]:
import pandas as pd
from collections import defaultdict
df_iata = pd.read_csv(iata_csv)
df_clean = df_iata.dropna(subset=['iata_code'])
iata_mapping = defaultdict(list)
for _, row in df_clean.iterrows():
    if pd.notna(row['municipality']):
        city = row['municipality'].lower().strip()
        code = row['iata_code']
        if code not in iata_mapping[city]:
            iata_mapping[city].append(code)
iata_mapping['tehran'] = ['IKA']
print("-" * 30)
print(f"Loaded {len(df_clean)} airports.")
print(f"Mapped {len(iata_mapping)} cities.")
print(f"Test Tehran: {iata_mapping.get('tehran')}") 
print(f"Test London: {iata_mapping.get('london')}") 
print(f"Test Dubai:  {iata_mapping.get('dubai')}")

### Country to currency code

In [ ]:

import pandas as pd
import pandas as pd
df_curr = pd.read_csv(currency_csv)
currency_mapping = {}
for _, row in df_curr.iterrows():
    if pd.notna(row['Country']) and pd.notna(row['Currency_Code']):
        country = row['Country'].lower().strip()
        code = row['Currency_Code']
        if country not in currency_mapping:
            currency_mapping[country] = code



print("-" * 30)
print(f"Mapped {len(currency_mapping)} countries.")
print(f"Test LATVIA: {currency_mapping.get('latvia')}") 
print(f"Test USA:   {currency_mapping.get('usa')}")

## 3. Normalizing city and country names

Users type `newyork`, `New York`, `نیویورک`. `difflib.get_close_matches` with a
tuned cutoff handles the first two; the LLM handles transliteration before the
tool is called.

In [ ]:
import pandas as pd
import re
from collections import defaultdict

def clean_text_strict(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'\s*\(.*?\)', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.strip().lower()


print(" Processing Currency Data")
currency_mapping = {}

try:
    df_curr = pd.read_csv(currency_csv)
    for _, row in df_curr.iterrows():
        clean_country = clean_text_strict(row['Country'])
        code = row['Currency_Code']
        
        if clean_country and pd.notna(code):
            if clean_country not in currency_mapping:
                currency_mapping[clean_country] = code
except Exception as e:
    print(f"Warning: Could not load currency CSV: {e}")

currency_overrides = {
    'iran': 'IRR',
    'usa': 'USD', 'united states': 'USD', 'united states of america': 'USD',
    'uk': 'GBP', 'united kingdom': 'GBP',
    'uae': 'AED', 'united arab emirates': 'AED',
    'china': 'CNY',
    'euro': 'EUR', 'europe': 'EUR',
    'turkey': 'TRY'
}
currency_mapping.update(currency_overrides)

ALL_COUNTRIES = list(currency_mapping.keys())
print(f" Currency Ready: {len(ALL_COUNTRIES)} countries mapped.")



print("\n Processing IATA Airport Data")
iata_mapping = defaultdict(list)

try:
    df_iata = pd.read_csv(iata_csv)
    

    df_iata = df_iata.dropna(subset=['iata_code'])
    df_iata = df_iata[df_iata['type'] != 'closed']
    df_iata = df_iata[df_iata['type'] != 'heliport']

    for _, row in df_iata.iterrows():
        clean_city = clean_text_strict(row['municipality'])
        code = row['iata_code']
        
        if clean_city:
        
            if code not in iata_mapping[clean_city]:
                iata_mapping[clean_city].append(code)
                
except Exception as e:
    print(f" Warning: Could not load airport CSV: {e}")


CITY_OVERRIDES = {
    "tehran": ["IKA"],      
    "dubai": ["DXB", "DWC"],      
    "london": ["LHR", "LGW", "STN", "LCY", "LTN"],
    "paris": ["CDG", "ORY", "BVA"],
    "new york": ["JFK", "EWR", "LGA"],
    "istanbul": ["IST", "SAW"],
    "moscow": ["SVO", "DME", "VKO"],
    "tokyo": ["NRT", "HND"],
    "rome": ["FCO", "CIA"],
    "milan": ["MXP", "LIN", "BGY"]
}

print(" Applying Manual Overrides for Major Hubs")
for city, codes in CITY_OVERRIDES.items():
    iata_mapping[city] = codes 

ALL_CITIES = list(iata_mapping.keys())
print(f" IATA Ready: {len(ALL_CITIES)} cities mapped.")

# Final test
print("\n Final Check:")
print(f"   Tehran: {iata_mapping.get('tehran')}") 
print(f"   Dubai:  {iata_mapping.get('dubai')}") 
print(f"   London: {len(iata_mapping.get('london'))} airports")

### Checking the mapping

In [ ]:
print(f"Test Dubai:  {iata_mapping.get('dubai')}")

### The fuzzy matcher

In [ ]:
def fuzzy_find(query: str, possibilities: list, cutoff=0.8):
    
    query_clean = query.lower().strip()
    
    
    matches = difflib.get_close_matches(query_clean, possibilities, n=1, cutoff=cutoff)
    
    if matches:
        return matches[0] 
    return None

## 4. The FAQ knowledge base

In [ ]:
import json
import pandas as pd
import os

faq_file_path = 'FAQ.js'

if os.path.exists(faq_file_path):
    with open(faq_file_path, 'r', encoding='utf-8') as f:
        content = f.read()
        
     
        if "[" in content and "]" in content:
            start = content.find("[")
            end = content.rfind("]") + 1
            json_data = content[start:end]
            faq_data = json.loads(json_data)
        else:
            faq_data = json.loads(content)
    
    print(f" Successfully loaded {len(faq_data)} FAQ items.")
    print(f" First item sample: {faq_data[0]}")

  

## 5. Parsing a travel guide with LlamaParse

Extracting a PDF travel guide page by page into a dataframe, for the itinerary
planner to retrieve from.

In [ ]:
import nest_asyncio
nest_asyncio.apply()
from llama_parse import LlamaParse
import pandas as pd
import os
import datetime

os.getenv('LLAMA_CLOUD_API_KEY')


parser = LlamaParse(
    result_type="markdown",  
    verbose=True,
    language="en",
    system_prompt="""
    You are a precise text extraction assistant.
        Task:
        1. Identify the primary COUNTRY and CITY discussed in the text.
        2. Format the output as follows:
           - First line: Markdown Header 1 with Country and City (e.g., "# Argentina Buenos Aires" or "# General").
           - Subsequent lines: The EXACT, FULL original text of the page.
           
        CRITICAL RULES:
        - DO NOT summarize the text.
        - DO NOT omit any paragraphs.
        - YOU MUST RETURN THE ORIGINAL CONTENT AFTER THE HEADER.
        - **IMAGE CONTENT:** If an image contains words(selectable or not ), write that word out exactly. DO NOT describe the image (e.g., do not write "image of a menu"), just write the text inside it.
        - **NO CONTENT:** Output "# NO_CONTENT" **ONLY** if the page is completely empty or contains images that have **NO text** inside them.
        - **Header Format:** First line must be "# Country City" (or "# General").
        - **Output:** Return the header followed by the full transcribed text.
        - 
    """
)


book_filename = "World Travel Book.pdf"

if os.path.exists(book_filename):
    print(f" Parsing '{book_filename}'... (This may take a minute)")
    
    documents = parser.load_data(book_filename)
    
    creation_timestamp = os.path.getctime(book_filename)
    creation_date = datetime.datetime.fromtimestamp(creation_timestamp).strftime('%Y-%m-%d')
    book_source_name = "World Travel Book"

    data_list = []
    
    for i, doc in enumerate(documents, start=1):
        
        raw_text = doc.text.strip()
        
        
        chunk_title = "General"
        body_text = raw_text 

       
        lines = raw_text.split('\n')
        
        if len(lines) > 0:
            first_line = lines[0].strip()
            
           
            if first_line.startswith("#"):
                
                chunk_title = first_line.lstrip("#").strip()
                
                
                if len(lines) > 1:
                    body_text = "\n".join(lines[1:]).strip()
                else:
                    body_text = ""  
            
           
            if "NO_CONTENT_HERE" in first_line or "NO_CONTENT_HERE" in body_text:
                body_text = ""

     
        data_list.append({
            "text": body_text,      
            "page_number": i,
            "title": chunk_title,   
            "creation_date": creation_date,
            "source": book_source_name
        })
    
    df_book = pd.DataFrame(data_list)
    
    print(f" Parsing Complete. Created DataFrame with {len(df_book)} rows.")
    print("Sample Output:")
    display(df_book.head())
    
   



### Caching the parsed output

LlamaParse is a paid API call, so the result is written to CSV and reloaded from
there. Re-running the notebook does not re-parse.

In [ ]:
output_filename = str(DATA / 'parsed_travel_book.csv')
df_book.to_csv(output_filename, index=False)

In [ ]:

output_file = str(DATA / 'parsed_travel_book.csv')
df_book = pd.read_csv(output_file)
print(f" Successfully saved parsed data to '{output_file}'")

### Spot-checking a page

In [ ]:
page_number = 26
full_text = df_book[df_book["page_number"] == page_number]["text"].values[0]
print(full_text)

## 6. Vector store for the guide

In [ ]:
import lancedb
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

registry = get_registry()
func = registry.get("huggingface").create(name="BAAI/bge-small-en-v1.5")

db = lancedb.connect("./lancedb_store")
print("Database connected/created.")



class FAQSchema(LanceModel):
    question: str = func.SourceField() 
    answer: str
    vector: Vector(func.ndims()) = func.VectorField() 

class BookSchema(LanceModel):
    text: str = func.SourceField()
    page_number: int
    source: str
    title: str  
    vector: Vector(func.ndims()) = func.VectorField()

print(" Schemas defined using BAAI/bge-small-en-v1.5")

### Indexing the pages

In [ ]:
import json
import os
import pandas as pd

faq_file_path = 'FAQ.js'
faq_data = []

if os.path.exists(faq_file_path):
   
    with open(faq_file_path, 'r', encoding='utf-8') as f:
        content = f.read()
        start = content.find("[")
        end = content.rfind("]") + 1
        if start != -1 and end != -1:
            json_str = content[start:end]
            faq_data = json.loads(json_str)
        else:
            faq_data = json.loads(content)
    print(f" Loaded {len(faq_data)} FAQ items.")
    



if faq_data:
    print(" Populating FAQ Table")
    faq_records = [{"question": item["question"], "answer": item["answer"]} for item in faq_data]
    
    faq_table = db.create_table("faq", schema=FAQSchema, mode="overwrite")
    faq_table.add(faq_records)
    print(f"FAQ Table created with {len(faq_records)} records.")


if 'df_book' in locals() and not df_book.empty:
    print("\n Populating Book Table with Empty Page Handling")
    

    df_book['text'] = df_book['text'].fillna("").astype(str)
    df_book['title'] = df_book['title'].fillna("General").astype(str)
    df_book['source'] = df_book['source'].fillna("Unknown").astype(str)
    
    book_records = []

    NO_CONTENT_SENTENCE = "This page contains no text content."

    for _, row in df_book.iterrows():
        
       
        p_num = -1
        try:
            val = row.get('page_number', -1)
            if pd.notna(val):
                p_num = int(float(val))
        except:
            p_num = -1

      
        raw_text = row['text'].strip()
        
 
        if (not raw_text) or (raw_text == "# NO_CONTENT") or (len(raw_text) < 5):
            final_text = NO_CONTENT_SENTENCE
           
            final_title = "No Content"
        else:
            final_text = raw_text
            final_title = row['title']

        book_records.append({
            "text": final_text,          
            "page_number": p_num,
            "source": row['source'],
            "title": final_title
        })

    book_table = db.create_table("book", schema=BookSchema, mode="overwrite")
    book_table.add(book_records)
    print(f" Book Table created with {len(book_records)} records.")



print(" TESTING SEARCH")
print("="*30)


q = "documents for international travel"
q = "documents for international travel"
if 'faq_table' in locals():
    res = faq_table.search(q).limit(1).to_pandas()
    if not res.empty:
        print(f" Question Matched: {res.iloc[0]['question']}")
        print(f" Answer: {res.iloc[0]['answer']}")  

if 'book_table' in locals():
    res = book_table.search("Paris").limit(1).to_pandas()
    if not res.empty:
        print(f" Book Context Found: ...{res.iloc[0]['text'][:50]}...")
    else:
        print(" No Book context found.")

## 7. `search_flights` — Amadeus

The most involved tool. Fuzzy-match both cities, expand each to all its airports,
query every origin-destination pair, and merge the offers.

The multi-airport expansion is what makes it work on real input. A user saying
"Tokyo to New York" does not know or care that this is six route queries; without
the expansion the tool would pick one arbitrary pair and report whatever that pair
happened to have.

Note the empty-result handling: an Amadeus 200 with an empty list is not an error,
it means that specific route has no offers on that date. The tool distinguishes
this from a failure and continues to the next pair.

In [ ]:
from langchain_core.tools import tool
from amadeus import Client, ResponseError
import difflib
import os

@tool
def search_flights(origin_city: str, destination_city: str, date: str) -> str:
    """
    Searches for flights between two cities on a specific date using Amadeus API.
    Uses Fuzzy Matching to identify cities and checks ALL associated airports.

    Args:
        origin_city (str): The name of the departure city (e.g. "Tehran").
        destination_city (str): The name of the arrival city (e.g. "Paris").
        date (str): The date of the flight in 'YYYY-MM-DD' format.
    """
    print(f"\n DEBUG START: Searching {origin_city} -> {destination_city} on {date}")

    try:
        
        match_origin = difflib.get_close_matches(origin_city.lower().strip(), ALL_CITIES, n=1, cutoff=0.2)
        match_dest = difflib.get_close_matches(destination_city.lower().strip(), ALL_CITIES, n=1, cutoff=0.2)
    
        print(f"    Fuzzy Match Raw Results:")
        print(f"      Origin: {match_origin}")
        print(f"      Dest:   {match_dest}")
        
        real_origin = match_origin[0] if match_origin else None
        real_dest = match_dest[0] if match_dest else None
        
        if not real_origin:
            print("    Error: Origin city fuzzy match failed.")
            return f"Error: Could not find city matching '{origin_city}'."
        if not real_dest:
            print("    Error: Destination city fuzzy match failed.")
            return f"Error: Could not find city matching '{destination_city}'."
            
       
        origin_val = iata_mapping[real_origin]
        dest_val = iata_mapping[real_dest]

        
        if isinstance(origin_val, str):
            origin_codes_list = [origin_val]
        else:
            origin_codes_list = origin_val

        if isinstance(dest_val, str):
            dest_codes_list = [dest_val]
        else:
            dest_codes_list = dest_val
            
        

        print(f"    IATA Codes Selected:")
        print(f"      Origin: {origin_codes_list}") 
        print(f"      Dest:   {dest_codes_list}")   

        all_flight_offers = []

       
        print("    Starting API Calls loop...")
        
        for o_code in origin_codes_list:
            for d_code in dest_codes_list:
                print(f"       Testing pair: {o_code} -> {d_code} ...", end=" ")
                try:
                    response = amadeus.shopping.flight_offers_search.get(
                        originLocationCode=o_code,
                        destinationLocationCode=d_code,
                        departureDate=date,
                        adults=1,
                        max=3
                    )
                    
                    if response.data:
                        count = len(response.data)
                        print(f" Found {count} offers.")
                        all_flight_offers.extend(response.data)
                    else:
                        print(" Success (200 OK) but NO flights returned (Empty List).")
                        
                except ResponseError as error:
                    
                    print(f"\n       AMADEUS API ERROR:")
                    print(f"         Status: {error.response.status_code}")
                    print(f"         Code: {error.code}")
                    print(f"         Message: {error.response.body}")
                    continue 
                except Exception as e:
                    print(f"\n       PYTHON ERROR: {e}")
                    continue

       
        if not all_flight_offers:
            print("    Loop finished. Result: No flights found in total.")
            return f"No flights found from {real_origin} to {real_dest} on {date}."

        print(f"    Loop finished. Total offers collected: {len(all_flight_offers)}")
        
        sorted_flights = sorted(all_flight_offers, key=lambda x: float(x['price']['total']))
        top_results = sorted_flights[:3]

        results_text = []
        for offer in top_results:
            price = offer['price']['total']
            currency = offer['price']['currency']
            itineraries = offer['itineraries'][0]['segments']
            
            airline_code = itineraries[0]['carrierCode']
            dep_code = itineraries[0]['departure']['iataCode']
            arr_code = itineraries[-1]['arrival']['iataCode']
            departure_time = itineraries[0]['departure']['at']
            duration = offer['itineraries'][0]['duration']
            stops = len(itineraries) - 1
            
            flight_info = (
                f" Airline: {airline_code} | Route: {dep_code} -> {arr_code}\n"
                f"    Price: {price} {currency}\n"
                f"    Departs: {departure_time} | Duration: {duration} | Stops: {stops}"
            )
            results_text.append(flight_info)

        return (f"Found flights for {real_origin} (checked: {origin_codes_list}) "
                f"to {real_dest} (checked: {dest_codes_list}):\n\n" + 
                "\n\n".join(results_text))

    except Exception as e:
        print(f"    CRITICAL FUNCTION ERROR: {e}")
        return f"An error occurred during flight search: {str(e)}"

### Testing it: Tehran to Dubai

`IKA -> DXB` returns 3 offers; `IKA -> DWC` returns 200 OK with an empty list and
is dropped. Prices in EUR, with duration and stop count.

In [ ]:
print(search_flights.invoke({"origin_city": "tehran", "destination_city": "Dubai", "date": "2026-03-22"}))

## 8. `search_hotels` — Amadeus with a web fallback

Amadeus hotel coverage is uneven. Where it has inventory (Dubai) the tool returns
real offers with addresses and prices; where it does not (Canberra) it falls back
to a Tavily web search and returns TripAdvisor-style results with ratings.

Both paths are visible in the evaluation scenarios, and the fallback is why the
Canberra request succeeds at all.

In [ ]:
import os
import difflib
from langchain_core.tools import tool
from amadeus import Client, ResponseError
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

@tool
def search_hotels(destination_city: str, check_in_date: str, check_out_date: str, budget: str = None) -> str:
    """
    Searches for hotels in a specific city for a given date range using Amadeus API.
    Iterates through ALL associated IATA codes (like flight search).
    Uses a caching mechanism to preserve Address and Rating data from the first API call.
    
    Arg:
     
        destination_city (str): The name of the arrival city (e.g. "Paris").
        check_in_date (str): The check-in date in 'YYYY-MM-DD' format.
        check_out_date (str): The check-out date in 'YYYY-MM-DD' format.
        budget (str): Maximum budget per night (optional).
    """
    try:
        matches = difflib.get_close_matches(destination_city.lower().strip(), ALL_CITIES, n=1, cutoff=0.6)
        real_city = matches[0] if matches else None
        
        if not real_city:
            raise Exception(f"City {destination_city} not found in database.")
            
        city_codes_list = iata_mapping[real_city]
        
        amadeus_results = []

        for code in city_codes_list:
            try:
                
                hotels_response = amadeus.reference_data.locations.hotels.by_city.get(
                    cityCode=code
                )
                
                if not hotels_response.data:
                    continue

                
                hotel_details_map = {}
                for h in hotels_response.data:
                    h_id = h.get('hotelId')
                    address = h.get('address', {}).get('lines', [])
                    rating = h.get('rating', 'N/A')
                    
                    full_address = ", ".join(address) if address else "Address N/A"
                    
                    hotel_details_map[h_id] = {
                        "address": full_address,
                        "rating": rating
                    }

                
                top_10_hotels = hotels_response.data[:10]
                hotel_ids = [h['hotelId'] for h in top_10_hotels]
                hotel_ids_str = ",".join(hotel_ids)

                response = amadeus.shopping.hotel_offers_search.get(
                    hotelIds=hotel_ids_str,
                    checkInDate=check_in_date,
                    checkOutDate=check_out_date,
                    adults=1,
                    bestRateOnly=True
                )

                if response.data:
                    for offer in response.data[:5]:
                        hotel_obj = offer.get('hotel', {})
                        h_id = hotel_obj.get('hotelId')
                        name = hotel_obj.get('name', 'Unknown Hotel')
                        
                   
                        cached_info = hotel_details_map.get(h_id, {})
                        address = cached_info.get('address', 'Address N/A')
                        rating = cached_info.get('rating', 'N/A')
                        
                        price_val = "N/A"
                        currency = ""
                        if offer.get('offers'):
                            price_opt = offer['offers'][0]
                            price_val = price_opt.get('price', {}).get('total', 'N/A')
                            currency = price_opt.get('price', {}).get('currency', '')
                        
                        desc = (f"Hotel: {name}\n"
                                f"Rating: {rating}\n"
                                f"Address: {address}\n"
                                f"Price: {price_val} {currency}")
                        amadeus_results.append(desc)
                    
                    return f"Amadeus Hotel Offers in {real_city.title()} ({code}):\n\n" + "\n\n".join(amadeus_results)
            
            except ResponseError:
                continue
            except Exception:
                continue

        raise Exception("No hotel offers found for any associated city code.")

    except Exception as e:
        try:
            query = f"hotels in {destination_city} from {check_in_date} to {check_out_date}"
            if budget:
                query += f" with budget {budget}"
            
            response = tavily.search(query=query, max_results=3)
            
            web_results = []
            for r in response.get('results', []):
                title = r.get('title', 'Hotel Result')
                url = r.get('url', '#')
                content = r.get('content', '')[:200]
                
                formatted_item = (
                    f"Title: {title}\n"
                    f"Link: {url}\n"
                    f"Info: {content}"
                )
                web_results.append(formatted_item)
            
            if web_results:
                return f"Web Search Results for {destination_city}:\n\n" + "\n\n".join(web_results)
            else:
                return "No hotels found via API or Web."
                
        except Exception as tavily_error:
            return f"Error: Both APIs failed. {tavily_error}"

### Testing it: Dubai

In [ ]:
print(search_hotels.invoke({"destination_city": "Dubai", "check_in_date": "2026-03-01", "check_out_date": "2026-03-03"}))

## 9. `search_restaurants` — Tavily

Web search rather than a booking API. For large cities this returns curated lists
(Time Out, Condé Nast) rather than individual venues, which is the more useful
answer — a single restaurant recommendation for "restaurants in London" would be
close to arbitrary.

In [ ]:
from langchain_core.tools import tool
from tavily import TavilyClient
import os

tavily = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

@tool
def search_restaurants(destination: str) -> str:
    """
    Searches for the top-rated restaurants and local food spots in a destination.
    Returns a formatted list with names, descriptions, and links.

    Args:
        destination (str): The city or location name (e.g., "Paris").
    """
    try:
       
        query = f"top 5 best restaurants and authentic local food in {destination} with reviews and cuisine type"
        
   
        response = tavily.search(
            query=query, 
            search_depth="advanced", 
            max_results=2
        )
        
        results = []
      
        for i, res in enumerate(response.get('results', []), 1):
            title = res.get('title', 'Unknown Restaurant')
            url = res.get('url', '#')
            
            content = res.get('content', 'No description available.')[:500].strip() + "..."
            
          
            formatted_entry = (
                f"{i}. **{title}**\n"
                f"   - Description: {content}\n"
                f"   - More Info: {url}\n"
                f"   {'-' * 40}" 
            )
            results.append(formatted_entry)
            
        if not results:
            return f"No restaurants found in {destination}."
            
      
        final_output = f"Top Recommended Restaurants in {destination.title()}:\n\n" + "\n".join(results)
        return final_output
        
    except Exception as e:
        return f"Error searching restaurants: {e}"

### Testing it

In [ ]:
print(search_restaurants.invoke({"destination": "dubai"}))

## 10. `get_weather`

Current conditions and forecast by city and date. When the user asks about a range
("weather next week in Moscow") the *agent* decides to call this tool once per day
rather than the tool handling ranges — the looping logic lives in the model, not in
the code.

In [ ]:
import os
import ast
from langchain_core.tools import tool
from tavily import TavilyClient

tavily = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

@tool
def get_weather(destination: str, date: str) -> str:
    """
    Gets the weather forecast.
    Auto-detects if the result is raw data (dict/json) and formats it cleanly.
    
    Args:
        destination (str): The city or location name (e.g., "Paris").
        date (str): The date of the day  in 'YYYY-MM-DD' format.
    """
    try:
        query = f"weather forecast in {destination} on {date}"
        response = tavily.search(query=query, search_depth="basic", max_results=1)
        
        results = response.get('results', [])
        if not results:
            return f"No weather data found for {destination}."

        data = results[0]
        title = data.get('title', 'Weather Info')
        raw_content = data.get('content', 'No details available.')
        url = data.get('url', '#')
        
      
        final_details = ""
        
        try:
            
            parsed_data = ast.literal_eval(raw_content)
            
            if isinstance(parsed_data, dict) and 'current' in parsed_data:
                
                curr = parsed_data.get('current', {})
                condition = curr.get('condition', {}).get('text', 'N/A')
                temp = curr.get('temp_c', 'N/A')
                humidity = curr.get('humidity', 'N/A')
                wind = curr.get('wind_kph', 'N/A')
                last_update = curr.get('last_updated', 'N/A')
                
                final_details = (
                    f"Condition: {condition}, Temp: {temp}°C, "
                    f"Humidity: {humidity}%, Wind: {wind} km/h "
                    f"(Updated: {last_update})"
                )
            else:
                
                final_details = " ".join(raw_content.split())[:300]
                
        except (ValueError, SyntaxError):
           
            final_details = " ".join(raw_content.split())[:300]

 
        formatted_result = (
            f" -  Location: {destination.title()}\n"
            f" -  Date: {date}\n"
            f" -  Summary: {title}\n"
            f" -  Details: {final_details}\n"
            f" -  Link: {url}"
        )

        return formatted_result

    except Exception as e:
        return f"Error: {e}"

### Testing it

In [ ]:
print(get_weather.invoke({"destination": "Dubai", "date": "2026-02-13"}))

## 11. `get_currency_rate`

Country names to currency codes, then a live rate. The response includes a worked
example ("to buy 5 Qatari Riyal it would cost you AED 5.04") because a bare
exchange rate is harder to act on than a concrete conversion.

In [ ]:
from langchain_core.tools import tool
import difflib
import os
from tavily import TavilyClient


tavily = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

@tool
def get_currency_rate(origin_country: str, destination_country: str) -> str:
    """
    Gets the bidirectional exchange rate between two countries (e.g., 1 USD = ? AED).
    Returns a clean, two-line format similar to:
    1 USD = 3.67 AED
    1 AED = 0.27 USD

    Args:
        origin_country (str): Origin country (e.g. "USA").
        destination_country (str): Destination country (e.g. "UAE").
    """
    try:
        c1_input = origin_country.lower().strip()
        c2_input = destination_country.lower().strip()
        
        
        def resolve_country(name_input):
            if name_input in currency_mapping:
                return name_input
            matches = difflib.get_close_matches(name_input, ALL_COUNTRIES, n=1, cutoff=0.7)
            return matches[0] if matches else None

        real_origin = resolve_country(c1_input)
        real_dest = resolve_country(c2_input)
        
        
        missing = []
        if not real_origin: missing.append(origin_country)
        if not real_dest: missing.append(destination_country)
        
        if missing:
            return f"Error: Could not identify country: {', '.join(missing)}."

        curr1 = currency_mapping[real_origin]
        curr2 = currency_mapping[real_dest]
        
        if curr1 == curr2:
            return f"1 {curr1} = 1 {curr2} (Same Currency)"

        
        query = f"1 {curr1} to {curr2} and 1 {curr2} to {curr1} exchange rate"
        
        response = tavily.search(query=query, max_results=1)
        
        if response.get('results'):
            content = response['results'][0]['content']
            url = response['results'][0]['url']
            
            # Tidy the text for display
            clean_content = " ".join(content.split())

            return (
                f" **Exchange Rate ({curr1} ↔ {curr2})**\n"
                f"1 {curr1} = ... {curr2}\n"
                f"1 {curr2} = ... {curr1}\n"
                f"-------------------\n"
                f" **Data:** {clean_content}\n"
                f" **Source:** {url}"
            )
            
        return "Exchange rate data not found."

    except Exception as e:
        return f"System Error: {e}"

### Testing it

In [ ]:
print(get_currency_rate.invoke({"origin_country": "iran", "destination_country": "usa"}))

## 12. `search_faq`

Retrieval over the FAQ knowledge base. This is the tool that fails in evaluation
— see scenario 13 in the analysis doc.

In [ ]:
@tool
def search_faq(query: str) -> str:
    """
    Searches the travel agency's FAQ database for answers to common questions.
    
    Args:
        query (str): The user's question.
    """
    try:
      
        if 'faq_table' not in globals():
            return "Error: FAQ database is not loaded."

  
        results = faq_table.search(query).limit(2).to_pandas()
        
        if results.empty:
            return "No relevant FAQ found."
            
        answer_text = "Based on our FAQ:\n\n"
        for _, row in results.iterrows():
            answer_text += f"Q: {row['question']}\nA: {row['answer']}\n\n"
            
        return answer_text
    except Exception as e:
        return f"Database Error: {e}"

### Testing it

In [ ]:
print(search_faq.invoke({"query": "What documents do I need for international travel?"}))

## 13. `plan_trip`

Itinerary generation, retrieving from the parsed travel guide and combining it
with the destination, date range, and an optional interests parameter.

Its weakness is geographic: it will happily propose Mexico City and Cancún in a
five-day itinerary without accounting for the distance between them. The tool has
no notion of intra-destination travel time.

In [ ]:
from langchain_core.tools import tool
from datetime import datetime
import pandas as pd
from tavily import TavilyClient
import os


tavily = TavilyClient(api_key=os.environ.get("TAVILY_API_KEY"))

@tool
def plan_trip(destination: str, arrival_date: str, departure_date: str, interests: str = "") -> str:
    """
    Generates a comprehensive travel itinerary for a specific destination and date range.
    It combines internal knowledge (RAG) with real-time web search to create a day-by-day plan.

    Args:
        destination (str): The city or country to visit (e.g., "Paris", "Japan").
        arrival_date (str): Arrival date in 'YYYY-MM-DD' format.
        departure_date (str): Departure date in 'YYYY-MM-DD' format.
        interests (str, optional): User interests (e.g., "museums, food, hiking").
    """
    itinerary_output = []
    
    
    try:
        d1 = datetime.strptime(arrival_date, "%Y-%m-%d")
        d2 = datetime.strptime(departure_date, "%Y-%m-%d")
        delta = d2 - d1
        trip_days = delta.days + 1
        if trip_days < 1:
            trip_days = 1 
    except ValueError:
        return " Error: Dates must be in YYYY-MM-DD format."
    except Exception as e:
        trip_days = 3 
    
    header = f" **Trip Plan for {destination.title()} ({trip_days} Days)**\n" \
             f" Date: {arrival_date} to {departure_date}\n" \
             f" Interests: {interests if interests else 'General Sightseeing'}\n" \
             f"{'='*40}\n"
    itinerary_output.append(header)

  
    rag_content = ""
    try:
        if 'book_table' in globals():
          
            queries = [
                f"top tourist attractions in {destination}",
                f"best local food and restaurants in {destination}",
                f"cultural and historical facts about {destination}"
            ]
            if interests:
                queries.append(f"{interests} activities in {destination}")

            seen_texts = set() 
            
            for q in queries:
               
                results = book_table.search(q).limit(2).to_pandas()
                for _, row in results.iterrows():
                    text = row['text']
                  
                    if text not in seen_texts and len(text) > 50:
                        rag_content += f" *From Guide:* {text[:300]}...\n"
                        seen_texts.add(text)
            
            if rag_content:
                itinerary_output.append(f" **Insights from Knowledge Base:**\n{rag_content}\n{'-'*40}\n")
        else:
            itinerary_output.append(" Note: Local knowledge base not accessible. Relying on web search.\n")

    except Exception as e:
        itinerary_output.append(f" RAG Search Error: {e}\n")


    try:

        web_query = f"{trip_days} day itinerary for {destination} focusing on {interests} including hidden gems and restaurants"
        
        web_response = tavily.search(
            query=web_query, 
            search_depth="advanced", 
            max_results=3
        )
        
        web_results = []
        for res in web_response.get('results', []):
            title = res.get('title', 'Itinerary Suggestion')
            content = res.get('content', '')
            url = res.get('url', '#')
            
           
            clean_content = " ".join(content.split())[:500] + "..."
            
            web_results.append(
                f" **{title}**\n"
                f"    {clean_content}\n"
                f"    [Read Full Plan]({url})"
            )
            
        if web_results:
            itinerary_output.append(f" **Suggested Itineraries (Web Sources):**\n\n" + "\n\n".join(web_results))
        else:
            itinerary_output.append("No specific web itinerary found.")

    except Exception as e:
        itinerary_output.append(f" Web Search Error: {e}")

   
    final_output = "\n".join(itinerary_output)
    return final_output

### Testing it

In [ ]:
print(plan_trip.invoke({"destination": "paris", "arrival_date": "2026-03-13", "departure_date": "2026-03-17", "interests": "history" }))

## 14. The agent

`gpt-4o-mini` through an OpenAI-compatible endpoint, bound to all seven tools. The
system prompt injects the current date, which is what lets "next week" resolve to
an actual date instead of being passed through as a literal.

In [ ]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage
from datetime import datetime
from langchain_ollama import ChatOllama

llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["LLM_API_KEY"],  
    base_url="https://api.avalai.ir/v1",  
    temperature=0.1

)



tools_list = [
    search_flights,
    search_hotels,
    search_restaurants,
    get_weather,
    get_currency_rate,
    search_faq,
    plan_trip
]


llm_with_tools = llm.bind_tools(tools_list)


current_date = datetime.now().strftime("%Y-%m-%d")

SYS_PROMPT = f"""You are TravelBot, an expert AI travel assistant for an online travel agency.
Current Date: {current_date}

Your goal is to assist users with planning trips, booking flights/hotels, finding restaurants, checking weather, and answering travel questions.

### GUIDELINES:
1. **Always use the provided tools** to get real-time information. Do not hallucinate flight prices or weather.
2. **Date Handling:** If a user says "next Friday" or "tomorrow", calculate the specific date based on the Current Date ({current_date}) before calling a tool. Tools require 'YYYY-MM-DD' format.
3. **Flight/Hotel Search:**
   - Always map city names to IATA codes (e.g., Tehran -> IKA) internally, but you can speak naturally to the user.
   - If no flights/hotels are found, ask the user for flexible dates or alternative destinations.
4. **Currency:** When asked about prices in different currencies, use the 'get_currency_rate' tool.
5. **Trip Planning:** Use the 'plan_trip' tool for comprehensive itineraries.
6. **FAQ:** If the user asks general questions (visa, baggage, policies), use 'search_faq'.
7. **Language:** Communicate in English. Be polite, professional, and helpful.

### TOOLS AVAILABLE:
- search_flights: Find flights between cities.
- search_hotels: Find accommodation.
- search_restaurants: Find food/dining.
- get_weather: Check weather forecasts.
- get_currency_rate: Convert currencies.
- search_faq: Answer general questions using the knowledge base.
- plan_trip: Generate a full travel itinerary.
"""

print(f"LLM Initialized and {len(tools_list)} tools bound successfully.")

## 15. The graph

A standard tool-calling loop: the model node decides whether to call tools, the
`ToolNode` executes them, and control returns to the model until it answers
without requesting a tool.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing import Annotated, TypedDict
from langchain_core.messages import BaseMessage, SystemMessage
from langgraph.checkpoint.memory import MemorySaver 


class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

def reasoner(state: AgentState):
  
    return {"messages": [llm_with_tools.invoke([SystemMessage(content=SYS_PROMPT)] + state["messages"])]}


workflow = StateGraph(AgentState)

workflow.add_node("reasoner", reasoner)
workflow.add_node("tools", ToolNode(tools_list))

workflow.add_edge(START, "reasoner")
workflow.add_conditional_edges("reasoner", tools_condition)
workflow.add_edge("tools", "reasoner")


memory = MemorySaver() 


app = workflow.compile(checkpointer=memory)

print(" Agent Graph compiled WITH MEMORY successfully!")



### The compiled graph

In [ ]:

display(Image(app.get_graph().draw_mermaid_png()))


## 16. Evaluation: 14 conversational scenarios

Mixed Persian and English, testing entity extraction, relative date resolution,
multi-turn tool use, and parallel tool calls.

Highlights from the transcript:

- **Persian input, correct tools.** *"هوای تهران چطور هست الان؟"* → `get_weather(Tehran,
  2026-02-13)`, answered in Persian with temperature, humidity, and wind.
- **Multi-day forecasts by looping.** *"wether next week moscow"* → three separate
  `get_weather` calls, one per day. The agent inferred that a range needs
  repetition.
- **Parallel tool calls.** The Dubai request triggered `search_flights`,
  `search_hotels`, `get_weather`, and `get_currency_rate` in a single turn.
- **Cross-lingual consistency.** The same FAQ question in Persian and English
  produced substantively the same answer.
- **A retrieval failure the model papered over.** The Persian travel-documents
  question retrieved an FAQ entry about *language barriers*. The LLM ignored the
  irrelevant context and answered correctly from general knowledge — a good
  outcome that hides a bad retrieval, and exactly the kind of thing that makes
  RAG systems hard to evaluate end-to-end.

The full scenario-by-scenario write-up, with a quality rating and criticism for
each, is in [`docs/travelbot-evaluation.md`](../docs/travelbot-evaluation.md).

In [ ]:
import uuid

def chat_cli():
    print(" Welcome to TravelBot! (Type 'exit' to quit, 'clear' to reset) ")
    print("-" * 60)
    
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}
    
    while True:
        try:
            user_input = input("\n You: ").strip()
            
            if user_input.lower() in ["exit", "quit"]:
                print(" Goodbye!")
                break
            
            if user_input.lower() == "clear":
                thread_id = str(uuid.uuid4())
                config = {"configurable": {"thread_id": thread_id}}
                print(" Conversation cleared (New Thread Started).")
                continue
            
            if not user_input:
                continue

            print(" TravelBot is thinking...", end="", flush=True)

            inputs = {"messages": [("user", user_input)]}
            
            final_response = ""
            
            for event in app.stream(inputs, config=config, stream_mode="values"):
                message = event["messages"][-1]
                
                if hasattr(message, "tool_calls") and message.tool_calls:
                    for tc in message.tool_calls:
                        print(f"\n     Calling Tool: {tc['name']}")
                        print(f"    Input: {tc['args']}")
                
                elif message.type == "tool":
                    print(f"    Output: {message.content[:200]}...")
                
                elif message.type == "ai" and not message.tool_calls:
                    final_response = message.content

            print(f"\r TravelBot: {final_response}\n")

        except Exception as e:
            print(f"\n Error: {e}")

if __name__ == "__main__":
    chat_cli()